# MCMC Diagnostics: Convergence, Mixing, and Troubleshooting

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why MCMC diagnostics are essential** — samples from a non-converged chain give wrong answers.
2. Read **trace plots** and distinguish well-mixed chains ("hairy caterpillars") from pathological ones.
3. Compute and interpret **$\hat{R}$ (R-hat)**, the potential scale reduction factor.
4. Understand **effective sample size (ESS)** and why raw sample count overstates information.
5. Identify **divergent transitions** in HMC/NUTS and fix them with **non-centered parameterisations**.
6. Apply a systematic **diagnostic checklist** to any MCMC output.
7. Troubleshoot non-convergence using a structured workflow.

## Prerequisites

- [03_mcmc_samplers.ipynb](03_mcmc_samplers.ipynb) — Metropolis-Hastings, HMC, NUTS
- [04_pymc_intro.ipynb](04_pymc_intro.ipynb) — Building models in PyMC
- [Module 02](../02_distributions/02_continuous_distributions.ipynb) — Normal distribution, variance

In [ ]:
import sys, os, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        "sudo apt-get update -qq && sudo apt-get install -y -qq "
        "libcairo2-dev libpango1.0-dev && pip install -q manim ipython==8.21.0 "
        "pymc arviz"
    )

_miktex_bin = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64"
if _miktex_bin.exists() and str(_miktex_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + str(_miktex_bin)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:
# PyTensor Windows fix: skip C compilation
os.environ["PYTENSOR_FLAGS"] = "device=cpu,floatX=float64,cxx="
    import pymc as pm
    import arviz as az
    print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")
except Exception:
    pm, az = None, None
    print("PyMC/ArviZ not available — PyMC cells will be skipped.")

sys.path.insert(0, os.path.abspath("../../src"))
from amstats.plotting import apply_style

apply_style()


class Cfg:
    root = Path("../../").resolve()
    gif_dir = root / "media" / "gifs"
    has_latex: bool = (
        shutil.which("latex") is not None or shutil.which("pdflatex") is not None
    )

    def __init__(self):
        self.gif_dir.mkdir(parents=True, exist_ok=True)

    def apply_manim_config(self):
        from manim import config as mcfg

        mcfg.format = "gif"

    def math_text(self, expr, **kwargs):
        from manim import MathTex, Text

        if self.has_latex:
            return MathTex(expr, **kwargs)
        return Text(expr, **kwargs)

    def save_gifs(self, clean=True):
        local_media = Path("media")
        found = list(local_media.rglob("*.gif")) if local_media.exists() else []
        if not found:
            print("  No new GIFs to save.")
            return
        for gif in found:
            dest = self.gif_dir / gif.name
            shutil.copy2(gif, dest)
            print(f"  ✓ media/gifs/{gif.name}")
        if clean:
            for sub in ("videos", "images", "Tex"):
                d = local_media / sub
                if d.exists():
                    shutil.rmtree(d, ignore_errors=True)
            print("  Cleaned up local temp render files (kept media/jupyter/).")


cfg = Cfg()
rng = np.random.default_rng(42)

---

## 1. Why Diagnostics Matter

MCMC gives you **samples** that, *in the limit*, come from the posterior distribution $p(\theta \mid y)$. But "in the limit" is doing a lot of heavy lifting. In practice, your chain has run for a **finite** number of iterations, and there is no guarantee that those samples faithfully represent the posterior.

If the chain has **not converged**, your posterior summaries — means, credible intervals, predictions — are **wrong**. Not just imprecise. *Wrong.* They reflect wherever the chain happened to wander, not the true posterior.

Common failure modes:

| Failure | What happens | Consequence |
|---------|-------------|-------------|
| **Not converged** | Chain still exploring, hasn't found the high-probability region | Posterior summaries are arbitrary |
| **Poor mixing** | Chain gets stuck in one mode or moves too slowly | Biased summaries, underestimated uncertainty |
| **Multimodality** | Chain finds one mode but misses others | Missing entire regions of the posterior |
| **Numerical issues** | Sampler makes large integration errors (divergences) | Biased samples, often near boundaries |

The fundamental problem: **you cannot tell from the samples alone whether they are from the right distribution.** Diagnostics are indirect tests — they can flag problems but cannot prove correctness. That said, failing diagnostics is a **reliable indicator of trouble**.

> **Rule:** Always check diagnostics before interpreting MCMC output. Never skip this step.

---

## 2. Trace Plots

A **trace plot** shows the sampled parameter value on the $y$-axis against the iteration number on the $x$-axis. It is the most basic — and often the most informative — diagnostic.

### What good chains look like

A well-mixed, converged chain looks like a **"hairy caterpillar"**:
- **Stationary:** no upward or downward trend; the chain fluctuates around a stable level.
- **Well-mixed:** rapid, random fluctuations; the chain moves freely across the parameter space.
- **Overlapping chains:** if you run multiple chains from different starting points, they should overlap and be indistinguishable.

### What bad chains look like

- **Trends:** the chain drifts upward or downward — it hasn't found the stationary distribution.
- **Separated modes:** different chains settle in different regions and don't mix.
- **Slow wandering:** the chain moves sluggishly, taking many steps to traverse the parameter space (high autocorrelation).
- **Stickiness:** the chain gets stuck at one value for many iterations (proposal rejection rate too high, or discrete modes).

Let us simulate both scenarios.

In [ ]:
# --- Simulate good and bad chains ---
n_iter = 2000

# GOOD chains: independent draws from the target (Normal(3, 1))
# (In reality MCMC samples are correlated, but well-tuned NUTS is close to this.)
good_chains = rng.normal(loc=3.0, scale=1.0, size=(4, n_iter))

# BAD chain 1: random walk with tiny step size (poor mixing / high autocorrelation)
bad_chain_slow = np.zeros(n_iter)
bad_chain_slow[0] = -2.0  # start far from the mode
for t in range(1, n_iter):
    bad_chain_slow[t] = bad_chain_slow[t - 1] + rng.normal(0, 0.05)

# BAD chain 2: stuck in a mode, then jumps
bad_chain_stuck = np.concatenate([
    rng.normal(-2, 0.1, n_iter // 2),  # stuck near -2
    rng.normal(3, 0.1, n_iter // 2),    # jumps to 3
])

# BAD chain 3: strong trend (hasn't converged)
bad_chain_trend = np.cumsum(rng.normal(0.01, 0.3, n_iter))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Good chains
ax = axes[0, 0]
for i in range(4):
    ax.plot(good_chains[i], alpha=0.7, linewidth=0.5)
ax.set_title('Good: "Hairy caterpillar" — well-mixed, stationary', fontsize=11)
ax.set_ylabel(r"$\theta$")
ax.set_xlabel("Iteration")

# Bad: slow random walk
ax = axes[0, 1]
ax.plot(bad_chain_slow, color="C3", linewidth=0.7)
ax.set_title("Bad: Slow random walk — tiny step size", fontsize=11)
ax.set_ylabel(r"$\theta$")
ax.set_xlabel("Iteration")

# Bad: mode switching
ax = axes[1, 0]
ax.plot(bad_chain_stuck, color="C1", linewidth=0.7)
ax.set_title("Bad: Stuck in modes — chain trapped, then jumps", fontsize=11)
ax.set_ylabel(r"$\theta$")
ax.set_xlabel("Iteration")

# Bad: trend
ax = axes[1, 1]
ax.plot(bad_chain_trend, color="C2", linewidth=0.7)
ax.set_title("Bad: Strong trend — not converged", fontsize=11)
ax.set_ylabel(r"$\theta$")
ax.set_xlabel("Iteration")

plt.suptitle("Trace Plot Gallery: Recognising Good and Bad Chains", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The top-left panel is what you want: four chains that overlap completely, with rapid, random fluctuations. The other three panels show common pathologies. Learn to recognise them at a glance — trace plots are the first thing you should look at after sampling.

### ArviZ trace plots

In practice, you use `az.plot_trace()` which combines the trace plot with a KDE of the marginal posterior. Let us run a simple PyMC model and inspect its traces.

In [ ]:
if pm is not None:
    # Simple model: estimate the mean of Normal data
    observed_data = rng.normal(loc=5.0, scale=2.0, size=100)

    with pm.Model() as simple_model:
        mu = pm.Normal("mu", mu=0, sigma=10)
        sigma = pm.HalfNormal("sigma", sigma=5)
        likelihood = pm.Normal("obs", mu=mu, sigma=sigma, observed=observed_data)
        trace_good = pm.sample(2000, chains=4, random_seed=42, return_inferencedata=True)

    az.plot_trace(trace_good, figsize=(12, 5))
    plt.suptitle("ArviZ Trace Plot — Well-Behaved Model", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("PyMC not available — skipping.")

In the ArviZ trace plot, the left column shows the KDE of the posterior marginals (all four chains should overlap), and the right column shows the trace for each chain. For this simple model, everything looks healthy: the caterpillars are hairy and the KDEs coincide.

---

## 3. $\hat{R}$ — The Potential Scale Reduction Factor

Visual inspection is important but subjective. **$\hat{R}$** ("R-hat"), introduced by Gelman and Rubin (1992), provides a **quantitative convergence diagnostic**.

### The idea

Run $M$ chains from **different starting points**, each of length $N$ (after discarding warmup). If all chains have converged to the same stationary distribution, then:
- The **within-chain variance** $W$ should be similar to
- The **between-chain variance** $B$.

If the chains have not converged, $B$ will be inflated (chains are in different places), making $\hat{R}$ large.

### Formal definition

Let $\theta_{m,n}$ denote the $n$-th draw from chain $m$. Define:

**Chain means:**
$$\bar{\theta}_m = \frac{1}{N} \sum_{n=1}^{N} \theta_{m,n}, \qquad \bar{\theta} = \frac{1}{M} \sum_{m=1}^{M} \bar{\theta}_m$$

**Between-chain variance:**
$$B = \frac{N}{M - 1} \sum_{m=1}^{M} (\bar{\theta}_m - \bar{\theta})^2$$

**Within-chain variance:**
$$W = \frac{1}{M} \sum_{m=1}^{M} s_m^2, \qquad s_m^2 = \frac{1}{N-1} \sum_{n=1}^{N} (\theta_{m,n} - \bar{\theta}_m)^2$$

**Pooled variance estimate:**
$$\widehat{\text{var}}^+ = \frac{N-1}{N} W + \frac{1}{N} B$$

This is a weighted average: when chains haven't converged, $B$ is large, inflating $\widehat{\text{var}}^+$ relative to $W$.

**$\hat{R}$:**
$$\boxed{\hat{R} = \sqrt{\frac{\widehat{\text{var}}^+}{W}}}$$

### Interpretation

- $\hat{R} \approx 1.0$: chains have converged (within-chain and between-chain variation are comparable).
- $\hat{R} > 1.01$: **warning** — chains may not have converged. Investigate.
- $\hat{R} > 1.1$: **serious problem** — do not trust the results.

The modern threshold is $\hat{R} < 1.01$ (Vehtari et al., 2021), stricter than the original $< 1.1$.

### From-scratch implementation

In [ ]:
def rhat_from_scratch(chains):
    """
    Compute R-hat from a 2D array of chains.

    Parameters
    ----------
    chains : array, shape (M, N)
        M chains, each of length N.

    Returns
    -------
    rhat : float
    """
    M, N = chains.shape

    # Chain means
    chain_means = chains.mean(axis=1)        # shape (M,)
    overall_mean = chain_means.mean()

    # Between-chain variance
    B = N / (M - 1) * np.sum((chain_means - overall_mean) ** 2)

    # Within-chain variance
    chain_vars = chains.var(axis=1, ddof=1)  # shape (M,)
    W = chain_vars.mean()

    # Pooled variance estimate
    var_plus = (N - 1) / N * W + B / N

    return np.sqrt(var_plus / W)


# --- Test on our simulated good chains ---
rhat_good = rhat_from_scratch(good_chains)
print(f"R-hat for good chains: {rhat_good:.4f}  (should be close to 1.0)")

# --- Test on bad chains: different starting points, poor mixing ---
bad_chains = np.zeros((4, n_iter))
starting_points = [-5, 0, 5, 10]
for i, start in enumerate(starting_points):
    bad_chains[i, 0] = start
    for t in range(1, n_iter):
        bad_chains[i, t] = bad_chains[i, t - 1] + rng.normal(0, 0.05)

rhat_bad = rhat_from_scratch(bad_chains)
print(f"R-hat for bad chains:  {rhat_bad:.4f}  (should be >> 1.0)")

In [ ]:
# Visualise the converged vs. non-converged chains side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for i in range(4):
    ax.plot(good_chains[i], alpha=0.7, linewidth=0.5)
ax.set_title(f"Converged chains — $\\hat{{R}}$ = {rhat_good:.4f}", fontsize=12)
ax.set_xlabel("Iteration")
ax.set_ylabel(r"$\theta$")

ax = axes[1]
for i in range(4):
    ax.plot(bad_chains[i], alpha=0.7, linewidth=0.5)
ax.set_title(f"Non-converged chains — $\\hat{{R}}$ = {rhat_bad:.4f}", fontsize=12)
ax.set_xlabel("Iteration")
ax.set_ylabel(r"$\theta$")

plt.suptitle("$\\hat{R}$: Quantifying Convergence", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### ArviZ R-hat

In practice, use `az.rhat()` which implements the improved rank-normalised $\hat{R}$ from Vehtari et al. (2021).

In [ ]:
if az is not None and pm is not None:
    rhat_vals = az.rhat(trace_good)
    print("ArviZ R-hat for the simple Normal model:")
    print(rhat_vals)
else:
    print("ArviZ not available — skipping.")

---

## 4. Effective Sample Size (ESS)

### The problem: autocorrelation

MCMC samples are **not independent**. Each sample depends on the previous one — this is inherent to the Markov chain construction. Consecutive samples are **positively autocorrelated**: knowing $\theta_t$ tells you something about $\theta_{t+1}$.

Why does this matter? If you have 4,000 samples but they are highly correlated, you might have only 200 "effective" independent samples worth of information. Using $n = 4000$ in standard error calculations would dramatically understate your uncertainty.

### Autocorrelation

The **autocorrelation at lag $t$** is:

$$\rho_t = \frac{\text{Cov}(\theta_n, \theta_{n+t})}{\text{Var}(\theta_n)}$$

For a well-mixing chain, $\rho_t$ decays rapidly toward zero. For a poorly-mixing chain, it stays high for many lags.

### Effective Sample Size

The **ESS** estimates how many independent samples your chain is equivalent to:

$$\boxed{\text{ESS} = \frac{n_{\text{samples}}}{1 + 2\sum_{t=1}^{\infty} \rho_t}}$$

The denominator is the **integrated autocorrelation time** $\tau = 1 + 2\sum \rho_t$. If consecutive samples are nearly independent ($\rho_t \approx 0$ for $t \geq 1$), then $\tau \approx 1$ and $\text{ESS} \approx n$. If autocorrelation is high, $\tau$ is large and ESS is much smaller than $n$.

### Rules of thumb

| ESS | Interpretation |
|-----|---------------|
| $> 1000$ | Excellent — reliable posterior summaries |
| $> 400$ | Adequate for most purposes |
| $100 - 400$ | Marginal — posterior means OK, tail quantiles unreliable |
| $< 100$ | Insufficient — do not trust the results |

### From-scratch: autocorrelation and ESS

In [ ]:
def autocorrelation(chain, max_lag=100):
    """Compute autocorrelation of a 1D chain up to max_lag."""
    n = len(chain)
    mean = chain.mean()
    var = chain.var()
    acf = np.zeros(max_lag + 1)
    for t in range(max_lag + 1):
        if t >= n:
            break
        acf[t] = np.mean((chain[: n - t] - mean) * (chain[t:] - mean)) / var
    return acf


def ess_from_scratch(chain, max_lag=100):
    """
    Estimate effective sample size using the initial positive sequence estimator.

    We sum autocorrelations until we hit a negative pair (Geyer's criterion)
    to avoid summing noisy estimates of high-lag autocorrelations.
    """
    acf = autocorrelation(chain, max_lag)
    n = len(chain)

    # Sum autocorrelations in pairs; stop when a pair sum is negative
    tau = 1.0
    for t in range(1, max_lag, 2):
        pair_sum = acf[t] + (acf[t + 1] if t + 1 <= max_lag else 0)
        if pair_sum < 0:
            break
        tau += 2 * pair_sum

    return n / tau


# --- Compare ESS for good vs. bad chains ---
ess_good = ess_from_scratch(good_chains[0])
ess_slow = ess_from_scratch(bad_chain_slow)

print(f"Good chain (n={n_iter}):  ESS = {ess_good:.0f}")
print(f"Slow chain (n={n_iter}):  ESS = {ess_slow:.0f}")
print(f"\nThe slow chain's {n_iter} samples are worth only ~{ess_slow:.0f} independent draws!")

In [ ]:
# --- Autocorrelation plots ---
max_lag = 80
acf_good = autocorrelation(good_chains[0], max_lag)
acf_slow = autocorrelation(bad_chain_slow, max_lag)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

ax = axes[0]
ax.bar(range(max_lag + 1), acf_good, width=1.0, alpha=0.7, color="C0")
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_title(f"Good chain — ESS = {ess_good:.0f}", fontsize=12)
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")

ax = axes[1]
ax.bar(range(max_lag + 1), acf_slow, width=1.0, alpha=0.7, color="C3")
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_title(f"Slow chain — ESS = {ess_slow:.0f}", fontsize=12)
ax.set_xlabel("Lag")

plt.suptitle("Autocorrelation: Good vs. Poor Mixing", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

The good chain's autocorrelation drops to zero almost immediately — consecutive samples are nearly independent. The slow chain's autocorrelation remains high for dozens of lags, meaning the chain is barely exploring new territory with each step.

### ArviZ ESS

In [ ]:
if az is not None and pm is not None:
    ess_vals = az.ess(trace_good)
    print("ArviZ ESS (bulk and tail) for the simple Normal model:")
    print(ess_vals)
    print()

    # ArviZ also reports bulk and tail ESS separately
    ess_bulk = az.ess(trace_good, method="bulk")
    ess_tail = az.ess(trace_good, method="tail")
    print("Bulk ESS:")
    print(ess_bulk)
    print("\nTail ESS:")
    print(ess_tail)
else:
    print("ArviZ not available — skipping.")

ArviZ reports **bulk ESS** (for the central tendency of the posterior) and **tail ESS** (for the 5th and 95th percentiles). The tail ESS is often lower because extreme quantiles need more samples to estimate accurately. Both should be $> 400$.

---

## 5. Divergent Transitions (HMC/NUTS Specific)

### What are divergent transitions?

The NUTS sampler (the default in PyMC) uses **Hamiltonian dynamics** to propose moves. It simulates the trajectory of a particle moving through the posterior landscape, using a numerical integrator (the leapfrog method).

A **divergent transition** occurs when the numerical integrator makes a **large error** — the simulated trajectory diverges from the true Hamiltonian trajectory. PyMC detects this by checking if the Hamiltonian (total energy) changes too much during a step.

### What they mean

Divergences are not random numerical noise. They almost always indicate that **the posterior geometry is pathological** in some region — typically:

1. **Funnel shapes** in hierarchical models (very narrow high-curvature regions)
2. **Sharp ridges or boundaries** in the posterior
3. **Near-degenerate parameters** that the sampler cannot navigate

Divergences mean the sampler **could not explore part of the posterior**, so your samples are biased — they under-represent the problematic region.

### The classic example: Neal's funnel

The canonical example is a hierarchical model that creates a "funnel" in the joint posterior:

$$\log \sigma \sim \mathcal{N}(0, 3)$$
$$x \sim \mathcal{N}(0, \sigma)$$

When $\sigma$ is large, $x$ ranges widely. When $\sigma$ is small, $x$ is constrained near zero. The joint distribution looks like a funnel — the neck is hard for HMC to navigate.

### The fix: non-centered parameterisation (NCP)

Instead of sampling $x \sim \mathcal{N}(0, \sigma)$ directly (**centered parameterisation**), we introduce an auxiliary variable:

$$x_{\text{raw}} \sim \mathcal{N}(0, 1), \qquad x = \sigma \cdot x_{\text{raw}}$$

Now the sampler explores $x_{\text{raw}}$ (a standard normal — easy geometry) and $\sigma$ independently. The funnel disappears in the $(x_{\text{raw}}, \sigma)$ space.

Let us demonstrate this with a concrete hierarchical model.

In [ ]:
if pm is not None:
    # --- Centered parameterisation (will produce divergences) ---
    J = 8  # number of groups
    observed_effects = rng.normal(loc=3.0, scale=2.0, size=J)
    effect_se = np.ones(J) * 1.5  # known standard errors

    with pm.Model() as centered_model:
        # Hyperpriors
        mu = pm.Normal("mu", mu=0, sigma=10)
        tau = pm.HalfNormal("tau", sigma=5)

        # Group effects — centered parameterisation
        theta = pm.Normal("theta", mu=mu, sigma=tau, shape=J)

        # Likelihood
        obs = pm.Normal("obs", mu=theta, sigma=effect_se, observed=observed_effects)

        trace_centered = pm.sample(
            2000, chains=4, random_seed=42, return_inferencedata=True,
            target_accept=0.8  # default — may produce divergences
        )

    n_divs_centered = trace_centered.sample_stats["diverging"].sum().values
    print(f"Centered model: {n_divs_centered} divergent transitions")
else:
    print("PyMC not available — skipping.")

In [ ]:
if pm is not None:
    # --- Non-centered parameterisation (fixes divergences) ---
    with pm.Model() as noncentered_model:
        # Hyperpriors (same as before)
        mu = pm.Normal("mu", mu=0, sigma=10)
        tau = pm.HalfNormal("tau", sigma=5)

        # Non-centered: sample raw offsets, then transform
        theta_raw = pm.Normal("theta_raw", mu=0, sigma=1, shape=J)
        theta = pm.Deterministic("theta", mu + tau * theta_raw)

        # Likelihood (same)
        obs = pm.Normal("obs", mu=theta, sigma=effect_se, observed=observed_effects)

        trace_noncentered = pm.sample(
            2000, chains=4, random_seed=42, return_inferencedata=True,
            target_accept=0.8
        )

    n_divs_nc = trace_noncentered.sample_stats["diverging"].sum().values
    print(f"Non-centered model: {n_divs_nc} divergent transitions")
    print(f"\nImprovement: {n_divs_centered} -> {n_divs_nc} divergences")
else:
    print("PyMC not available — skipping.")

In [ ]:
if pm is not None and az is not None:
    # Compare diagnostics
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Centered: plot tau vs. theta[0] to see the funnel
    ax = axes[0]
    tau_samples = trace_centered.posterior["tau"].values.flatten()
    theta0_samples = trace_centered.posterior["theta"].values[:, :, 0].flatten()
    diverging = trace_centered.sample_stats["diverging"].values.flatten()

    ax.scatter(tau_samples[~diverging], theta0_samples[~diverging],
               alpha=0.1, s=3, color="C0", label="Normal")
    if diverging.sum() > 0:
        ax.scatter(tau_samples[diverging], theta0_samples[diverging],
                   alpha=0.8, s=15, color="C3", marker="x", label="Divergent")
    ax.set_xlabel(r"$\tau$")
    ax.set_ylabel(r"$\theta_0$")
    ax.set_title(f"Centered (divergences = {n_divs_centered})")
    ax.legend()

    # Non-centered
    ax = axes[1]
    tau_nc = trace_noncentered.posterior["tau"].values.flatten()
    theta0_nc = trace_noncentered.posterior["theta"].values[:, :, 0].flatten()
    div_nc = trace_noncentered.sample_stats["diverging"].values.flatten()

    ax.scatter(tau_nc[~div_nc], theta0_nc[~div_nc],
               alpha=0.1, s=3, color="C0", label="Normal")
    if div_nc.sum() > 0:
        ax.scatter(tau_nc[div_nc], theta0_nc[div_nc],
                   alpha=0.8, s=15, color="C3", marker="x", label="Divergent")
    ax.set_xlabel(r"$\tau$")
    ax.set_ylabel(r"$\theta_0$")
    ax.set_title(f"Non-centered (divergences = {n_divs_nc})")
    ax.legend()

    plt.suptitle(r"Funnel Geometry: $\tau$ vs. $\theta_0$", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("PyMC/ArviZ not available — skipping.")

In the centered parameterisation, the divergent transitions (red crosses) cluster in the neck of the funnel where $\tau$ is small. The sampler cannot navigate this narrow region accurately. The non-centered parameterisation eliminates (or greatly reduces) divergences by making the geometry more uniform.

**Key takeaway:** Divergences are a gift — they tell you *where* the sampler is struggling. Don't just increase `target_accept`; reparameterise.

---

## 6. Manim Animation: Good vs. Bad Mixing

The following animation shows two trace plots side by side. On the left, a well-mixed chain fills the parameter space uniformly and its histogram matches the true posterior. On the right, a poorly-mixed chain gets stuck and drifts slowly, producing a distorted histogram.

In [ ]:
from manim import *

cfg.apply_manim_config()
math_text = cfg.math_text

from amstats.manim_utils import C

In [ ]:
%%manim -qm -v WARNING ChainMixing


class ChainMixing(Scene):
    """Good vs. bad MCMC mixing with trace plots and histograms."""

    def construct(self):
        rng_m = np.random.default_rng(123)

        # --- Title ---
        title = Text("MCMC Mixing: Good vs. Bad", font_size=32, color=C.LABEL)
        title.to_edge(UP, buff=0.3)
        self.play(Write(title), run_time=0.8)

        # --- Generate chains ---
        n_pts = 500
        # True posterior: N(3, 1)
        true_mu, true_sigma = 3.0, 1.0

        # Good chain: nearly independent draws
        good = rng_m.normal(true_mu, true_sigma, n_pts)

        # Bad chain: slow random walk starting far away
        bad = np.zeros(n_pts)
        bad[0] = -1.0
        for i in range(1, n_pts):
            bad[i] = bad[i - 1] + rng_m.normal(0, 0.08)

        # --- Layout: left = good, right = bad ---
        # Trace axes
        trace_width = 5.0
        trace_height = 2.0
        hist_width = 1.2

        left_center = LEFT * 3.2 + DOWN * 0.2
        right_center = RIGHT * 3.2 + DOWN * 0.2

        # Labels
        good_label = Text("Well-mixed", font_size=22, color=C.EMERALD)
        good_label.move_to(left_center + UP * 1.8)
        bad_label = Text("Poorly-mixed", font_size=22, color=C.SALMON)
        bad_label.move_to(right_center + UP * 1.8)
        self.play(Write(good_label), Write(bad_label), run_time=0.6)

        # --- Draw trace plots as polylines ---
        def make_trace_line(data, center, color, w, h, ymin, ymax):
            points = []
            for i, val in enumerate(data):
                x = center[0] - w / 2 + w * i / len(data)
                y_norm = (val - ymin) / (ymax - ymin)
                y = center[1] - h / 2 + h * y_norm
                points.append([x, y, 0])
            line = VMobject()
            line.set_points_smoothly(points[::3])  # subsample for smoothness
            line.set_stroke(color=color, width=1.5, opacity=0.9)
            return line

        # Axis rectangles
        for center in [left_center, right_center]:
            border = Rectangle(
                width=trace_width, height=trace_height,
                stroke_color=C.NEUTRAL, stroke_width=1
            ).move_to(center)
            self.add(border)

        ymin_good, ymax_good = -1, 7
        ymin_bad, ymax_bad = -3, 5

        trace_good = make_trace_line(
            good, left_center, C.EMERALD, trace_width, trace_height,
            ymin_good, ymax_good
        )
        trace_bad = make_trace_line(
            bad, right_center, C.SALMON, trace_width, trace_height,
            ymin_bad, ymax_bad
        )

        self.play(Create(trace_good), Create(trace_bad), run_time=3)
        self.wait(0.5)

        # --- Histograms below ---
        hist_center_y = -2.5
        hist_h = 1.2
        hist_w = 4.5
        n_bins = 25

        def make_histogram(data, center_x, color, true_mu, true_sigma, ymin, ymax):
            bars = VGroup()
            bins = np.linspace(ymin, ymax, n_bins + 1)
            counts, _ = np.histogram(data, bins=bins)
            max_count = counts.max() if counts.max() > 0 else 1
            bar_w = hist_w / n_bins
            for i, c in enumerate(counts):
                bar_h = hist_h * c / max_count
                if bar_h < 0.01:
                    continue
                bar = Rectangle(
                    width=bar_w * 0.85, height=bar_h,
                    fill_color=color, fill_opacity=0.6,
                    stroke_width=0.5, stroke_color=color
                )
                x_pos = center_x - hist_w / 2 + bar_w * (i + 0.5)
                bar.move_to([x_pos, hist_center_y - hist_h / 2 + bar_h / 2, 0])
                bars.add(bar)

            # True posterior curve
            curve_pts = []
            xs = np.linspace(ymin, ymax, 100)
            pdf_vals = np.exp(-0.5 * ((xs - true_mu) / true_sigma) ** 2)
            pdf_vals /= pdf_vals.max()  # normalise to plot height
            for x_val, p in zip(xs, pdf_vals):
                x_pos = center_x - hist_w / 2 + hist_w * (x_val - ymin) / (ymax - ymin)
                y_pos = hist_center_y - hist_h / 2 + hist_h * p
                curve_pts.append([x_pos, y_pos, 0])
            curve = VMobject()
            curve.set_points_smoothly(curve_pts)
            curve.set_stroke(color=C.PERIWINKLE, width=2.5)
            return bars, curve

        hist_label = Text("Histogram vs. true posterior", font_size=18, color=C.LABEL)
        hist_label.move_to([0, hist_center_y + hist_h / 2 + 0.3, 0])
        self.play(Write(hist_label), run_time=0.5)

        bars_good, curve_good = make_histogram(
            good, -3.2, C.EMERALD, true_mu, true_sigma, ymin_good, ymax_good
        )
        bars_bad, curve_bad = make_histogram(
            bad, 3.2, C.SALMON, true_mu, true_sigma, ymin_bad, ymax_bad
        )

        self.play(
            FadeIn(bars_good), Create(curve_good),
            FadeIn(bars_bad), Create(curve_bad),
            run_time=2
        )

        # Verdict labels
        match_label = Text("Matches!", font_size=20, color=C.EMERALD)
        match_label.move_to([-3.2, hist_center_y - hist_h / 2 - 0.35, 0])
        mismatch_label = Text("Distorted!", font_size=20, color=C.SALMON)
        mismatch_label.move_to([3.2, hist_center_y - hist_h / 2 - 0.35, 0])
        self.play(Write(match_label), Write(mismatch_label), run_time=0.7)

        self.wait(2)

The animation makes the consequence of poor mixing concrete: the well-mixed chain's histogram (green) aligns with the true posterior (blue curve), while the poorly-mixed chain's histogram (red) is shifted and narrower — it gives a **wrong answer** for both the location and spread of the posterior.

---

## 7. The Diagnostic Checklist

Before interpreting **any** MCMC output, run through this checklist:

| Check | Tool | Good | Bad |
|-------|------|------|-----|
| Trace plots | `az.plot_trace()` | Hairy caterpillars; chains overlap | Trends, separated modes, stickiness |
| $\hat{R}$ | `az.rhat()` | $< 1.01$ | $> 1.01$ |
| ESS (bulk) | `az.ess()` | $> 400$ | $< 100$ |
| ESS (tail) | `az.ess(method='tail')` | $> 400$ | $< 100$ |
| Divergences | `pm.sample()` warnings | 0 | Any |
| Posterior predictive | `az.plot_ppc()` | Simulated data matches observed data | Systematic misfit |

Let us run the full checklist on our well-behaved model.

In [ ]:
if pm is not None and az is not None:
    print("=" * 60)
    print("MCMC DIAGNOSTIC CHECKLIST")
    print("=" * 60)

    # 1. R-hat
    rhat_vals = az.rhat(trace_good)
    max_rhat = max(rhat_vals[v].values.max() for v in rhat_vals.data_vars)
    status = "PASS" if max_rhat < 1.01 else "FAIL"
    print(f"\n1. R-hat (max):     {max_rhat:.4f}  [{status}]")

    # 2. ESS bulk
    ess_bulk = az.ess(trace_good, method="bulk")
    min_ess_bulk = min(ess_bulk[v].values.min() for v in ess_bulk.data_vars)
    status = "PASS" if min_ess_bulk > 400 else "FAIL"
    print(f"2. ESS bulk (min):  {min_ess_bulk:.0f}     [{status}]")

    # 3. ESS tail
    ess_tail = az.ess(trace_good, method="tail")
    min_ess_tail = min(ess_tail[v].values.min() for v in ess_tail.data_vars)
    status = "PASS" if min_ess_tail > 400 else "FAIL"
    print(f"3. ESS tail (min):  {min_ess_tail:.0f}     [{status}]")

    # 4. Divergences
    n_divs = trace_good.sample_stats["diverging"].sum().values
    status = "PASS" if n_divs == 0 else "FAIL"
    print(f"4. Divergences:     {n_divs}        [{status}]")

    print("\n" + "=" * 60)
    all_pass = max_rhat < 1.01 and min_ess_bulk > 400 and min_ess_tail > 400 and n_divs == 0
    print(f"Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
    print("=" * 60)
else:
    print("PyMC/ArviZ not available — skipping.")

### Posterior predictive check

The final sanity check: generate fake data from the posterior and compare to the observed data. If the model is appropriate, the simulated data should look like the real data.

In [ ]:
if pm is not None and az is not None:
    with simple_model:
        pm.sample_posterior_predictive(trace_good, extend_inferencedata=True, random_seed=42)

    az.plot_ppc(trace_good, num_pp_samples=100, figsize=(10, 4))
    plt.title("Posterior Predictive Check")
    plt.tight_layout()
    plt.show()
else:
    print("PyMC/ArviZ not available — skipping.")

The thin blue lines (simulated replications from the posterior) should envelope the thick dark line (observed data). If they don't — if the model consistently over- or under-predicts certain regions — the model is misspecified, regardless of what the convergence diagnostics say.

---

## 8. Troubleshooting Non-Convergence

When diagnostics fail, work through this list **in order**:

### Step 1: Run longer / more chains

The simplest fix. Sometimes the chain just needs more time to explore. Double the number of draws and add more chains (e.g., 4 $\to$ 8). This is cheap to try and often sufficient.

### Step 2: Increase target acceptance rate

For HMC/NUTS, increasing `target_accept` (e.g., from 0.8 to 0.95 or 0.99) makes the sampler take smaller, more careful steps. This helps with moderate geometry problems but slows sampling.

```python
pm.sample(target_accept=0.95)
```

### Step 3: Reparameterise

This is the most important fix for **divergent transitions** in hierarchical models.

| Parameterisation | When to use |
|-----------------|-------------|
| **Centered:** $\theta \sim \mathcal{N}(\mu, \sigma)$ | Lots of data per group (likelihood dominates prior) |
| **Non-centered:** $\theta = \mu + \sigma \cdot z, \; z \sim \mathcal{N}(0, 1)$ | Few data per group (prior dominates likelihood) |

There is no single "correct" parameterisation — it depends on the data. Try both.

### Step 4: Use stronger (more informative) priors

Vague priors on scale parameters (e.g., $\sigma \sim \text{HalfCauchy}(25)$) can create pathological geometries. Replacing them with weakly informative priors (e.g., $\sigma \sim \text{HalfNormal}(5)$) regularises the posterior and often eliminates problems.

This is not "cheating" — it's encoding the reasonable prior knowledge that, say, a standard deviation of 100 is implausible for your data.

### Step 5: Rethink the model

If nothing else works, the model may be fundamentally misspecified or non-identifiable. Signs:
- Parameters are **highly correlated** in the posterior (the model is over-parameterised)
- The posterior is **multimodal** in ways that reflect model ambiguity, not genuine uncertainty
- The data simply cannot inform certain parameters (non-identifiability)

In this case, simplify the model or collect more informative data.

### Summary flowchart

```
Diagnostics fail?
  ├─ R-hat > 1.01 or ESS < 100
  │   ├─ Try more iterations / chains
  │   └─ Still failing? → Check for multimodality, rethink model
  │
  ├─ Divergent transitions
  │   ├─ Increase target_accept (0.95, 0.99)
  │   ├─ Reparameterise (centered ↔ non-centered)
  │   └─ Use stronger priors on scale parameters
  │
  └─ Posterior predictive check fails
      └─ Model is misspecified — revise the likelihood or add structure
```

---

## Exercises

**Exercise 5.1 ($\hat{R}$ from scratch).** Using the `rhat_from_scratch()` function above, investigate how $\hat{R}$ changes as the chain length increases. Generate 4 chains from a random walk (step size 0.3, different starting points: $-5, 0, 5, 10$) of lengths $N = 50, 100, 500, 2000, 10000$. Plot $\hat{R}$ vs. $N$. At what chain length does $\hat{R}$ drop below 1.01? What does this tell you about how long you need to run MCMC?

**Exercise 5.2 (ESS and thinning).** Generate a correlated chain by sampling from $\mathcal{N}(0, 1)$ via Metropolis-Hastings with proposal $\mathcal{N}(\theta_t, 0.1)$ (small step = high autocorrelation). Run 10,000 iterations. (a) Compute ESS. (b) Thin the chain by keeping every $k$-th sample for $k = 1, 5, 10, 50$. For each, compute ESS. Does thinning improve ESS *per retained sample*? (c) Is it better to thin or to use all samples? Explain.

**Exercise 5.3 (Divergences and parameterisation).** Implement Neal's funnel in PyMC:
$$\log \sigma \sim \mathcal{N}(0, 3), \qquad x \sim \mathcal{N}(0, \sigma)$$
(a) Sample with the centered parameterisation. Count divergences. (b) Reparameterise: $z \sim \mathcal{N}(0, 1)$, $x = \sigma \cdot z$. Count divergences. (c) Plot the samples in $(\log\sigma, x)$ space for both. Where do the divergences concentrate?

**Exercise 5.4 (Diagnostic automation).** Write a function `run_diagnostics(trace)` that takes an ArviZ InferenceData object and returns a dictionary with `{"rhat_max": ..., "ess_bulk_min": ..., "ess_tail_min": ..., "n_divergences": ..., "all_pass": True/False}`. Apply it to both the centered and non-centered hierarchical models from Section 5.

**Exercise 5.5 (Challenge: multimodal target).** Consider the mixture $p(\theta) = 0.5 \cdot \mathcal{N}(-3, 0.5) + 0.5 \cdot \mathcal{N}(3, 0.5)$. (a) Run Metropolis-Hastings with a small proposal ($\sigma = 0.3$) and a large proposal ($\sigma = 5$). For each, examine the trace plot and $\hat{R}$ (using 4 chains with different starts). Which explores both modes? (b) Why is multimodality fundamentally hard for standard MCMC? What strategies exist (tempering, mixture proposals)?

---

## Key Takeaways

1. **Never skip diagnostics.** MCMC samples from a non-converged chain give wrong answers — not just imprecise, *wrong*.

2. **Trace plots** are your first line of defence. Look for hairy caterpillars (good) vs. trends, modes, and stickiness (bad).

3. **$\hat{R}$** quantifies convergence by comparing between-chain and within-chain variance. Target: $\hat{R} < 1.01$.

4. **ESS** tells you how many independent samples your chain is worth. Autocorrelation reduces effective information. Target: ESS $> 400$.

5. **Divergent transitions** in HMC/NUTS signal pathological posterior geometry. The primary fix is **reparameterisation** (centered $\leftrightarrow$ non-centered), not just tuning.

6. **Posterior predictive checks** test whether the model itself is appropriate, beyond sampler convergence.

7. When troubleshooting: more iterations $\to$ increase `target_accept` $\to$ reparameterise $\to$ stronger priors $\to$ rethink the model.

---

This completes **Module 07: Bayesian Inference**. We have covered Bayes' theorem, conjugate priors, MCMC samplers, PyMC modelling, and now diagnostics.

**Next module:** [Module 08 -- Bayesian Regression](../08_bayesian_regression/01_bayesian_linear_regression.ipynb): We apply the Bayesian machinery to regression, combining prior knowledge with data to obtain full posterior distributions over model parameters.

In [ ]:
cfg.save_gifs(clean=True)